# 🌾 Weed Detection — Entraînement YOLOv8m
**Modèle** : YOLOv8 Medium | **Classes** : 10 espèces fusionnées | **GPU** : T4

---
### ✅ Checklist avant de lancer
- [ ] Settings → Accelerator → **GPU T4**
- [ ] Panneau droit → Add Data → upload ton dossier `data/split/` zippé
- [ ] Vérifie le chemin `DATASET_DIR` dans la cellule 2
- [ ] Lance les cellules **dans l'ordre**

## 1 — Vérification environnement

In [ ]:
import torch
import ultralytics
import numpy as np

print('=== Environnement ===')
print(f'Python      : 3.12')
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'NumPy       : {np.__version__}')
print(f'CUDA dispo  : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM        : {mem:.1f} GB')
else:
    print()
    print('⚠️  Pas de GPU détecté !')
    print('   → Settings → Accelerator → GPU T4')
    raise SystemExit('Active le GPU avant de continuer.')

## 2 — Chargement du dataset

In [ ]:
import os, shutil, zipfile, yaml
from pathlib import Path

# ─────────────────────────────────────────────────────────
#  ← Vérifie ce chemin dans le panneau droit sous /kaggle/input/
DATASET_INPUT = '/kaggle/input/weed-detection-split'
WORK_DIR      = '/kaggle/working'
DATASET_DIR   = f'{WORK_DIR}/dataset'
# ─────────────────────────────────────────────────────────

# Cas 1 : dataset uploadé comme ZIP
zip_candidates = list(Path(DATASET_INPUT).glob('*.zip'))
if zip_candidates:
    print(f'ZIP trouvé : {zip_candidates[0]}')
    print('Extraction...')
    with zipfile.ZipFile(zip_candidates[0], 'r') as z:
        z.extractall(DATASET_DIR)
    print('✅ Extrait')

# Cas 2 : dataset uploadé déjà extrait
else:
    print('Pas de ZIP trouvé — copie directe...')
    if Path(DATASET_DIR).exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(DATASET_INPUT, DATASET_DIR)
    print('✅ Copié')

# Vérification structure
print()
print('Structure du dataset :')
total_img = 0
for split in ['train', 'val', 'test']:
    img_dir = Path(DATASET_DIR) / split / 'images'
    lbl_dir = Path(DATASET_DIR) / split / 'labels'

    # cherche aussi dans des sous-dossiers si nécessaire
    if not img_dir.exists():
        candidates = list(Path(DATASET_DIR).rglob(f'{split}/images'))
        if candidates:
            img_dir = candidates[0]
            lbl_dir = candidates[0].parent / 'labels'

    if img_dir.exists():
        n_img = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
        n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        total_img += n_img
        print(f'  {split:5} → {n_img:>4} images | {n_lbl:>4} labels')
    else:
        print(f'  {split:5} → ❌ introuvable')

print(f'  TOTAL → {total_img} images')

## 3 — Préparation du data.yaml

In [ ]:
from pathlib import Path
import yaml, shutil

# Trouve le data.yaml dans le dataset (même dans des sous-dossiers)
yaml_candidates = list(Path(DATASET_DIR).rglob('data.yaml'))
if not yaml_candidates:
    raise FileNotFoundError('data.yaml introuvable dans le dataset !')

src_yaml = yaml_candidates[0]
work_yaml = Path(WORK_DIR) / 'data.yaml'

# Lecture
with open(src_yaml) as f:
    data = yaml.safe_load(f)

# Trouve la racine qui contient train/val/test
dataset_root = src_yaml.parent
for p in [dataset_root, dataset_root.parent]:
    if (p / 'train').exists():
        dataset_root = p
        break

# Mise à jour des chemins
data['path']  = str(dataset_root)
data['train'] = 'train/images'
data['val']   = 'val/images'
data['test']  = 'test/images'

# Sauvegarde dans /kaggle/working/ (modifiable)
with open(work_yaml, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print('data.yaml prêt :')
print(f'  path  : {data["path"]}')
print(f'  train : {data["train"]}')
print(f'  val   : {data["val"]}')
print(f'  test  : {data["test"]}')
print(f'  nc    : {data["nc"]}')
print(f'  names : {data["names"]}')

YAML_PATH = str(work_yaml)

## 4 — Entraînement YOLOv8m

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data     = YAML_PATH,
    epochs   = 150,
    imgsz    = 640,
    batch    = 16,
    patience = 30,            # plus patient qu'avant
    device   = 0,
    workers  = 2,

    # Optimisation
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,
    cos_lr       = True,      # cosine LR scheduler

    # Augmentation — importante avec ~500 images
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    degrees = 15.0,           # rotation (photos aériennes)
    flipud  = 0.5,            # flip vertical (utile vue du dessus)
    fliplr  = 0.5,
    mosaic  = 1.0,
    mixup   = 0.15,
    copy_paste = 0.1,         # copie-colle des objets entre images

    # Sauvegarde
    project = f'{WORK_DIR}/runs',
    name    = 'weed_v1',
    save    = True,
    plots   = True,
)

BEST_PT = f'{WORK_DIR}/runs/weed_v1/weights/best.pt'
print(f'\n✅ Entraînement terminé !')
print(f'   Meilleurs poids : {BEST_PT}')

## 5 — Courbes d'entraînement

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path(f'{WORK_DIR}/runs/weed_v1')

for plot in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    p = run_dir / plot
    if p.exists():
        print(f'── {plot} ──')
        display(Image(str(p), width=800))
        print()

## 6 — Évaluation sur le jeu de TEST

In [ ]:
from ultralytics import YOLO

best = YOLO(BEST_PT)
metrics = best.val(data=YAML_PATH, split='test', verbose=True)

map50    = metrics.box.map50
map5095  = metrics.box.map
prec     = metrics.box.mp
recall   = metrics.box.mr

print('\n' + '═'*45)
print('  RÉSULTATS SUR LE JEU DE TEST')
print('═'*45)
print(f'  mAP50      : {map50:.4f}   (objectif > 0.65)')
print(f'  mAP50-95   : {map5095:.4f}')
print(f'  Précision  : {prec:.4f}')
print(f'  Rappel     : {recall:.4f}')
print('═'*45)

if map50 >= 0.75:
    print('\n🟢 Excellent ! Modèle prêt.')
elif map50 >= 0.60:
    print('\n🟡 Bon résultat. Quelques ajustements possibles.')
elif map50 >= 0.45:
    print('\n🟠 Résultat moyen. Vérifie annotations et équilibre classes.')
else:
    print('\n🔴 mAP faible. Plus de données ou annotations à améliorer.')

## 7 — Visualisation des prédictions

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import glob, random

# 6 images de test aléatoires
test_imgs = glob.glob(f'{DATASET_DIR}/**/test/images/*.jpg', recursive=True)
test_imgs += glob.glob(f'{DATASET_DIR}/**/test/images/*.png', recursive=True)
sample = random.sample(test_imgs, min(6, len(test_imgs)))

preds = best.predict(
    source     = sample,
    conf       = 0.25,
    save       = True,
    project    = f'{WORK_DIR}/runs',
    name       = 'predictions',
    line_width = 2,
    exist_ok   = True,
)

pred_imgs = sorted(glob.glob(f'{WORK_DIR}/runs/predictions/*.jpg'))
for img_path in pred_imgs[:6]:
    display(Image(img_path, width=640))
    print()

## 8 — Téléchargement de best.pt

In [ ]:
import shutil
from pathlib import Path

dst = Path(WORK_DIR) / 'best.pt'
shutil.copy2(BEST_PT, dst)

size_mb = dst.stat().st_size / 1e6
print(f'✅ best.pt prêt  ({size_mb:.1f} MB)')
print()
print('Pour télécharger :')
print('  Panneau droit Kaggle → Output → best.pt → Download')
print()
print('Ensuite place le fichier dans :')
print('  weed-detection/models/best.pt')
print()
print('Puis lance localement :')
print('  python main.py --phase 3')